In [1]:
import pandas as pd

In [21]:
import os

mapping_file = pd.ExcelFile(os.path.join( 'metadata', 'vietnames_to_fiin.xlsx'))

In [22]:
df_bs = mapping_file.parse('Standard BS')[['Universal_caption', 'universal_code']]
df_is = mapping_file.parse('Standard IS')[['Universal_caption', 'universal_code']]
df_cf = mapping_file.parse('Standard CF')[['Universal_caption', 'universal_code']]

In [23]:
df_bs = df_bs.rename(columns={'Universal_caption': 'category_name', 'universal_code': 'category_code'})
df_is = df_is.rename(columns={'Universal_caption': 'category_name', 'universal_code': 'category_code'})
df_cf = df_cf.rename(columns={'Universal_caption': 'category_name', 'universal_code': 'category_code'})

df_map_fs = pd.concat([df_bs, df_is, df_cf], ignore_index=True)

df_map_fs.to_csv(os.path.join('metadata', 'map_category_code.csv'), index=False)


In [25]:
df_map_tm_bank = mapping_file.parse('fiin_TM_bank')
df_map_tm_sec = mapping_file.parse('fiin_TM_sec')
df_map_tm_corp = mapping_file.parse('fiin_TM_corp')


df_map_tm_bank['category_code'] = df_map_tm_bank['category_code'].apply(lambda x: "Bank_" + x)
df_map_tm_sec['category_code'] = df_map_tm_sec['category_code'].apply(lambda x: "Sec_" + x)
df_map_tm_corp['category_code'] = df_map_tm_corp['category_code'].apply(lambda x: "Corp_" + x)


map_share_sec = ~df_map_tm_sec['share_code'].isna()
map_share_corp = ~df_map_tm_corp['share_code'].isna()

df_map_tm_sec.loc[map_share_sec, 'category_code'] = df_map_tm_sec.loc[map_share_sec, 'share_code']
df_map_tm_corp.loc[map_share_corp, 'category_code'] = df_map_tm_corp.loc[map_share_corp, 'share_code']


df_map_tm_bank = df_map_tm_bank[['category_code', 'vi_caption', 'en_caption', 'parent_code']]
df_map_tm_sec = df_map_tm_sec[['category_code', 'vi_caption', 'en_caption', 'parent_code']]
df_map_tm_corp = df_map_tm_corp[['category_code', 'vi_caption', 'en_caption', 'parent_code']]

df_map_tm_sec.dropna(subset=['en_caption'], inplace=True)
df_map_tm_bank.dropna(subset=['en_caption'], inplace=True)
df_map_tm_corp.dropna(subset=['en_caption'], inplace=True)

df_tm_map = pd.concat([df_map_tm_bank, df_map_tm_sec, df_map_tm_corp], ignore_index=True)
df_tm_map.drop_duplicates(subset=['category_code'], inplace=True)
df_tm_map['en_caption'] = df_tm_map['en_caption'].apply(lambda x: "(Explaination) " + x)


In [27]:
df_map_fs

,category_name,category_code
0,A. CURRENT ASSETS,BS_100
1,(Securities) Financial Assets,BS_101
2,Cash and cash equivalents,BS_110
3,Cash,BS_111
4,(Bank) Deposits at the Central Bank,BS_112
...,...,...
341,Net cash flow from financing activities,CF_090
342,Net cash flow during the period,CF_100
343,Cash and cash equivalents at the beginning of ...,CF_110
344,Effects of exchange rate changes on foreign cu...,CF_120


In [28]:
df_tm_map.rename(columns={'en_caption': 'category_name'}, inplace=True)
df_tm_map = df_tm_map[['category_code', 'category_name']]
df_map = pd.concat([df_map_fs, df_tm_map], ignore_index=True)

df_map.to_csv(os.path.join('metadata', 'map_category_code.csv'), index=False)


In [29]:
df = pd.read_parquet('../../data/financial_statement_v3.parquet')

In [30]:
df

,data,stock_code,quarter,year,date_added,category_code
0,0.0,SSI,0.0,2010.0,2010-12-30,IS_003
1,0.0,SSI,0.0,2011.0,2011-12-30,IS_003
2,0.0,SSI,0.0,2012.0,2012-12-30,IS_003
3,0.0,SSI,0.0,2013.0,2013-12-30,IS_003
4,0.0,SSI,0.0,2014.0,2014-12-30,IS_003
...,...,...,...,...,...,...
380803,0.0,DGC,3.0,2024.0,2024-09-30,Corp_TM_157
380804,0.0,DGC,4.0,2024.0,2024-12-30,Corp_TM_157
380805,0.0,DGC,1.0,2025.0,2025-03-30,Corp_TM_157
380806,0.0,DGC,2.0,2025.0,2025-06-30,Corp_TM_157


In [ ]:
df = df.dropna(subset=['category_code'])

In [19]:
df.to_parquet('../../data/financial_statement_v3.parquet', index=False)

In [16]:
df_tm = pd.read_parquet('../../data/financial_statement_explaination_v3.parquet')

In [17]:
df = pd.concat([df, df_tm])

In [18]:
df

,data,stock_code,quarter,year,date_added,category_code
0,0.0,SSI,0.0,2010.0,2010-12-30,IS_003
1,0.0,SSI,0.0,2011.0,2011-12-30,IS_003
2,0.0,SSI,0.0,2012.0,2012-12-30,IS_003
3,0.0,SSI,0.0,2013.0,2013-12-30,IS_003
4,0.0,SSI,0.0,2014.0,2014-12-30,IS_003
...,...,...,...,...,...,...
218043,0.0,DGC,3.0,2024.0,2024-09-30,Corp_TM_157
218044,0.0,DGC,4.0,2024.0,2024-12-30,Corp_TM_157
218045,0.0,DGC,1.0,2025.0,2025-03-30,Corp_TM_157
218046,0.0,DGC,2.0,2025.0,2025-06-30,Corp_TM_157


In [5]:
df[df['category_code'] == 'IS_090']

,data,stock_code,quarter,year,date_added,universal_code_x,universal_code_y,category_code
720,0.000000,SSI,0.0,2010.0,2010-12-30,None,None,IS_090
721,0.000000,SSI,0.0,2011.0,2011-12-30,None,None,IS_090
722,0.000000,SSI,0.0,2012.0,2012-12-30,None,None,IS_090
723,0.000000,SSI,0.0,2013.0,2013-12-30,None,None,IS_090
724,-183.144865,SSI,0.0,2014.0,2014-12-30,None,None,IS_090
...,...,...,...,...,...,...,...,...
157085,-61.995634,DGC,3.0,2024.0,2024-09-30,None,None,IS_090
157086,-79.776846,DGC,4.0,2024.0,2024-12-30,None,None,IS_090
157087,-141.122729,DGC,1.0,2025.0,2025-03-30,None,None,IS_090
157088,-95.468665,DGC,2.0,2025.0,2025-06-30,None,None,IS_090
